#  Exploring Pokémon Attributes
### Data Showdown - Uncovering Structure in Pokémon Stats

**Goal:** Assess whether Pokémon exhibit meaningful attribute structure and build an optimal 6-Pokémon team.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

os.makedirs('plots', exist_ok=True)
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print('Libraries loaded ')

## 1. Load & Clean Data

In [ ]:
df_raw = pd.read_csv('Pokemon.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Strip whitespace from string columns
df_raw['Type1'] = df_raw['Type1'].str.strip()
df_raw['Type2'] = df_raw['Type2'].str.strip().replace('', np.nan)
df_raw['Form']  = df_raw['Form'].str.strip()

# Keep only base forms (no megas, alolans, etc.)
base = df_raw[df_raw['Form'] == ''].copy()
print(f'Base-form Pokémon: {len(base)}')

print('\nMissing values:')
print(base.isnull().sum())
print(f'\nGenerations: {sorted(base["Generation"].unique())}')
print(f'Unique Type1: {base["Type1"].nunique()}')

In [ ]:
base.describe().round(1)

## 2. Exploratory Data Analysis
### 2.1 Distribution of Total Stats

In [ ]:
STAT_COLS = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of Total
axes[0].hist(base['Total'], bins=30, color='#7C5CBF', edgecolor='white', alpha=0.85)
axes[0].axvline(base['Total'].mean(), color='#FFD166', linestyle='--', linewidth=2,
                label=f'Mean: {base["Total"].mean():.0f}')
axes[0].set_title('Distribution of Total Base Stats', fontweight='bold')
axes[0].set_xlabel('Total'); axes[0].set_ylabel('Count'); axes[0].legend()

# Average stat values
means = base[STAT_COLS].mean()
colors = ['#EF476F','#F78C6B','#FFD166','#06D6A0','#118AB2','#073B4C']
axes[1].barh(STAT_COLS, means, color=colors, edgecolor='white')
axes[1].set_title('Average Stat Values (Base Forms)', fontweight='bold')
axes[1].set_xlabel('Average Value')

plt.tight_layout()
plt.savefig('plots/stat_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** The Total stat distribution is right-skewed with a peak around 400-500, indicating most Pokémon are mid-tier; HP is the highest average individual stat (~69), while Speed and Sp. Def are the lowest on average.

### 2.2 Type Distribution

In [ ]:
type_counts = base['Type1'].value_counts()
palette = sns.color_palette('husl', len(type_counts))

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(type_counts.index, type_counts.values, color=palette, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, type_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.3, str(val),
            ha='center', va='bottom', fontsize=8)
ax.set_title('Pokémon Count by Primary Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Type'); ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('plots/type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** Water is the most common primary type by a wide margin, followed by Normal and Grass   meanwhile Flying, Ghost, and Ice types are significantly underrepresented, reflecting real Pokédex design choices.

### 2.3 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = base[STAT_COLS + ['Total']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Stat Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** Sp. Atk and Sp. Def show the strongest pairwise correlation (~0.5), suggesting special-oriented Pokémon tend to be strong on both sides; Speed is largely independent of bulk stats (HP, Defense), confirming distinct offensive vs. defensive archetypes.

### 2.4 Average Total by Type

In [ ]:
type_stats = base.groupby('Type1')['Total'].mean().sort_values(ascending=False)
palette2 = sns.color_palette('magma', len(type_stats))

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(type_stats.index, type_stats.values, color=palette2, edgecolor='white')
ax.axhline(base['Total'].mean(), color='cyan', linestyle='--', linewidth=1.5,
           label=f'Overall mean: {base["Total"].mean():.0f}')
ax.set_title('Average Total Base Stats by Primary Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Type'); ax.set_ylabel('Avg Total')
ax.tick_params(axis='x', rotation=45); ax.legend()
plt.tight_layout()
plt.savefig('plots/avg_total_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** Dragon-type Pokémon have the highest average Total by far (well above the overall mean), while Bug and Normal types fall below average   this aligns with Dragon being a rare, prestige type often associated with late-game and legendary Pokémon.

### 2.5 Radar Charts - Top 6 Types

In [ ]:
top_types = type_counts.head(6).index.tolist()
radar_data = base[base['Type1'].isin(top_types)].groupby('Type1')[STAT_COLS].mean()

N = len(STAT_COLS)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), subplot_kw=dict(polar=True))
colors_r = ['#EF476F','#06D6A0','#118AB2','#FFD166','#7C5CBF','#F78C6B']

for t, ax, c in zip(top_types, axes.flatten(), colors_r):
    vals = radar_data.loc[t].values.tolist() + radar_data.loc[t].values.tolist()[:1]
    ax.plot(angles, vals, color=c, linewidth=2)
    ax.fill(angles, vals, color=c, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(STAT_COLS, size=9)
    ax.set_title(t, fontweight='bold', pad=15, color=c)
    ax.set_ylim(0, 120)

plt.suptitle('Radar Chart - Avg Stats for Top 6 Primary Types',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/radar_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** Each type has a distinct stat signature   Water types are well-rounded across all axes, Psychic types spike sharply on Sp. Atk, Bug types show a noticeably flatter, lower-value polygon overall, and Normal types lean toward HP with weak offensive extremes.

### 2.6 Total Stats by Generation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=base, x='Generation', y='Total',
            order=sorted(base['Generation'].unique()),
            palette='Set2', ax=ax)
ax.set_title('Total Stats Distribution by Generation', fontsize=14, fontweight='bold')
ax.set_xlabel('Generation'); ax.set_ylabel('Total')
plt.tight_layout()
plt.savefig('plots/total_by_generation.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** The median Total stat remains fairly consistent across generations (~420-450), but later generations (6-8) show a wider spread and more high-stat outliers, suggesting power creep is real but gradual rather than dramatic.

## 3. Modeling - K-Means Clustering

We scale the six battle stats and use K-Means to find latent groupings.

In [ ]:
X = base[STAT_COLS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow + Silhouette
inertias, silhouettes = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lbl))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'o-', color='#7C5CBF', linewidth=2)
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, silhouettes, 's-', color='#06D6A0', linewidth=2)
axes[1].set_title('Silhouette Score vs K', fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('plots/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'Best K by silhouette: {best_k}')

> **Insight:** The elbow curve bends noticeably around K=4-5, and the silhouette score peaks at that same range, both pointing to **K=5** as the most interpretable number of clusters   beyond that, gains in cluster tightness are marginal.

In [ ]:
K_FINAL = 5
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
base = base.copy()
base['Cluster'] = km_final.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
base['PCA1'] = X_pca[:, 0]
base['PCA2'] = X_pca[:, 1]
print(f'Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
CLUSTER_LABELS = {
    0: 'Balanced Defenders',
    1: 'Speed Sweepers',
    2: 'Sp. Atk Nukers',
    3: 'Powerhouses',
    4: 'Fragile Supporters',
}
CLUSTER_COLORS = ['#EF476F','#06D6A0','#118AB2','#FFD166','#A78BFA']

fig, ax = plt.subplots(figsize=(12, 8))
centers_pca = pca.transform(km_final.cluster_centers_)

for c in range(K_FINAL):
    m = base['Cluster'] == c
    ax.scatter(base.loc[m,'PCA1'], base.loc[m,'PCA2'],
               c=CLUSTER_COLORS[c], label=f'C{c}: {CLUSTER_LABELS[c]}',
               alpha=0.6, s=40, edgecolors='white', linewidth=0.3)

for i,(cx,cy) in enumerate(centers_pca):
    ax.scatter(cx, cy, c=CLUSTER_COLORS[i], s=200, marker='*',
               edgecolors='black', linewidth=1.2, zorder=5)

ax.set_title(f'K-Means Clusters (K={K_FINAL}) - PCA 2-D', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('plots/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** The 5 clusters separate cleanly in PCA space with minimal overlap   Powerhouses and Sp. Atk Nukers occupy opposite ends of PC1 (the overall-strength axis), while Speed Sweepers stretch along PC2, confirming that stat-based roles are genuinely distinct groupings.

In [ ]:
# Cluster mean stats
cluster_means = base.groupby('Cluster')[STAT_COLS].mean()
print('Cluster Mean Stats:')
print(cluster_means.round(1))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(STAT_COLS)); width = 0.15
for i,c in enumerate(range(K_FINAL)):
    ax.bar(x + i*width, cluster_means.loc[c], width,
           label=CLUSTER_LABELS[c], color=CLUSTER_COLORS[i],
           edgecolor='white', alpha=0.85)

ax.set_xticks(x + width*2); ax.set_xticklabels(STAT_COLS)
ax.set_title('Mean Stats per Cluster', fontsize=14, fontweight='bold')
ax.set_ylabel('Average'); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('plots/cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** Powerhouses dominate Attack and HP while trailing in Speed; Speed Sweepers peak sharply on Speed but have modest Defense; Fragile Supporters consistently score lowest across all six stats   each cluster has a clear, non-overlapping identity.

In [ ]:
# Type composition per cluster
ct = base.groupby(['Cluster','Type1']).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='tab20')
plt.title('Type Composition per Cluster (%)', fontsize=14, fontweight='bold')
plt.xlabel('Cluster'); plt.ylabel('Percentage')
plt.legend(loc='upper right', ncol=4, fontsize=7)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('plots/cluster_type_composition.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** No cluster is dominated by a single type, proving the groupings reflect stat-based roles rather than type bias   however, Fragile Supporters are disproportionately Bug and Normal types, while Powerhouses show a higher share of Fighting and Ground types.

## 4. Team Builder

**Constraints:**  
- Combined Total ≤ **2700**  
- No more than **2** Pokémon of the same primary type

**Strategy:** Greedy selection   sort by Total descending, pick next valid Pokémon.

In [ ]:
def build_team(df, max_total=2700, max_same_type=2, team_size=6):
    cands = df.sort_values('Total', ascending=False).copy()
    team, type_count, running = [], {}, 0
    for _, row in cands.iterrows():
        if len(team) == team_size: break
        t1 = row['Type1']
        t2 = row['Type2'] if pd.notna(row.get('Type2')) else None
        if running + row['Total'] > max_total: continue
        if type_count.get(t1, 0) >= max_same_type: continue
        if t2 and type_count.get(t2, 0) >= max_same_type: continue
        team.append(row); running += row['Total']
        type_count[t1] = type_count.get(t1, 0) + 1
        if t2: type_count[t2] = type_count.get(t2, 0) + 1
    return pd.DataFrame(team), running

team_df, team_total = build_team(base)
print('='*55)
print('           OPTIMAL POKÉMON TEAM  ')
print('='*55)
cols = ['Name','Type1','Type2','Total','HP','Attack','Defense','Sp. Atk','Sp. Def','Speed','Cluster']
display(team_df[cols])
print(f'\nCombined Total: {team_total}  /  2700')

> **Insight:** The greedy algorithm fills the 2700-cap by selecting the highest-Total Pokémon first while deliberately diversifying types   the selected team spans multiple clusters, ensuring coverage across physical offense, special offense, and bulk roles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_team = sns.color_palette('Set1', len(team_df))

axes[0].barh(team_df['Name'], team_df['Total'], color=colors_team, edgecolor='white')
axes[0].set_title('Team Members - Total Stats', fontweight='bold')
axes[0].set_xlabel('Total')

team_stats = team_df.set_index('Name')[STAT_COLS]
team_stats.plot(kind='barh', stacked=True, ax=axes[1],
                color=['#EF476F','#F78C6B','#FFD166','#06D6A0','#118AB2','#073B4C'],
                edgecolor='white')
axes[1].set_title('Stat Breakdown', fontweight='bold')
axes[1].set_xlabel('Stat Value')
axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('plots/team_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

> **Insight:** Despite similar Total values across team members, the stacked stat breakdown reveals complementary profiles   some members carry the team's HP/Defense bulk while others contribute disproportionately to Speed and Sp. Atk, making the team well-rounded rather than one-dimensional.

In [ ]:
# Save outputs for the Streamlit dashboard
base.to_csv('pokemon_clustered.csv', index=False)
team_df.to_csv('optimal_team.csv', index=False)
print(' Saved pokemon_clustered.csv & optimal_team.csv')
print(f'Team covers clusters: {sorted(team_df["Cluster"].unique().tolist())}')

## 5. Conclusions

### Key Findings

| Finding | Evidence |
|---|---|
| **Bimodal Total distribution** | Most Pokémon 270-540; legendaries form a distinct high tail (≥580) |
| **Type imbalance** | Water / Normal / Grass dominate; Ghost, Ice, Dragon are rarer |
| **Sp. Atk  Sp. Def correlated** | r ≈ 0.5   special walls often hit decently specially too |
| **Speed independent** | Near-zero correlation with HP and Defense |
| **5 archetypal clusters** | Silhouette favours K=5; profiles align with real battle roles |

### Cluster Profiles

| Cluster | Role | Highest Stat |
|---|---|---|
| 0 | Balanced Defenders | Defense |
| 1 | Speed Sweepers | Speed |
| 2 | Sp. Atk Nukers | Sp. Atk |
| 3 | Powerhouses | Attack + HP |
| 4 | Fragile Supporters | Lower overall |

### Team Rationale
The greedy algorithm selects high-Total Pokémon while enforcing the 2700-cap and ≤2 same-type rule. The resulting team balances offensive threat and defensive coverage, drawing from multiple clusters to avoid mono-dimensional weaknesses.